# 06 — MEM Electricity | Automatic Download (raw landing)

Finds the newest **Informe de Desempeño — Anexos** workbook published by the Ministry of Energy and Mines (MEM)
and lands it, unchanged, in the raw archive `/Volumes/rd_economic_intelligence/bronze/mem_raw/ede_performance/`.

| Step | Rule |
|---|---|
| Discover | MEM website media index (WordPress REST API `wp-json/wp/v2/media`), files whose name contains `Desempeno` and `Anexos` and end in `.xlsx`; newest by publication date |
| Skip | The file is downloaded and its SHA-256 compared with every file already archived (shared helper `src/shared/acquisition.py`); identical content is never landed twice |
| Validate | The download must open as a workbook with the sheets `EDE's` and `Nuevo Regimen tarifario`; otherwise the task fails and nothing is written |
| Land | Append-only archive, one folder per arrival month: `ede_performance/<yyyy>/<mm>/<original name>`. Files are never overwritten or deleted. |
| Corrections | If MEM republishes a file under a name already archived but with different content, it is kept as a new version: `<name>__<yyyy-mm-dd>.xlsx` |
| Signal | Task value `downloaded` (true/false) lets the scheduled Job skip Bronze, Silver and Gold when nothing new was published |

Bronze (`06_MEM_Electricity_Bronze`) picks up new archive files incrementally with Auto Loader. **Status:** Development.

In [0]:
%pip install openpyxl==3.1.5 --quiet

In [0]:
dbutils.library.restartPython()

In [0]:
import sys, json, io, re, importlib
import openpyxl

nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
sys.path.insert(0, "/Workspace" + nb.rsplit("/", 2)[0] + "/shared")
import acquisition as ACQ
importlib.reload(ACQ)

LANDING = "/Volumes/rd_economic_intelligence/bronze/mem_raw/ede_performance/"
API = "https://mem.gob.do/wp-json/wp/v2/media?search=Anexos&per_page=100&orderby=date&order=desc&_fields=date,source_url"
REQUIRED_SHEETS = {"EDE's", "Nuevo Regimen tarifario"}

items = json.loads(ACQ.http_get(API, timeout=60))
cands = [(i["date"], i["source_url"]) for i in items
         if re.search(r"desempe(n|ñ)o", i["source_url"], re.I) and "anexo" in i["source_url"].lower() and i["source_url"].lower().endswith(".xlsx")]
if not cands:
    raise RuntimeError("No MEM annex workbook found in the media index")
pub_date, url = max(cands)
name = url.rsplit("/", 1)[-1]
print(f"Newest annex: {name} (published {pub_date[:10]})")

data = ACQ.http_get(url)
wb = openpyxl.load_workbook(io.BytesIO(data), read_only=True)
missing = REQUIRED_SHEETS - set(wb.sheetnames)
if missing:
    raise ValueError(f"Downloaded file failed the structural contract, missing sheets: {sorted(missing)}")

result = ACQ.land(data, name, LANDING)
# Read by the scheduled Job's condition task: downstream layers run only when a new file landed
dbutils.jobs.taskValues.set(key="downloaded", value=result["status"] == "DOWNLOADED")
dbutils.notebook.exit(json.dumps({**result, "published": pub_date[:10]}, ensure_ascii=False))